# Q-Mini English 512 — CPU-Only Resume Trainer (V9)

**This version is CPU-only by design.** It hides all CUDA devices before importing PyTorch, uses CPU FP32 training, and does not request a GPU, run GPU benchmarks, or call `nvidia-smi`. It reuses the same Google Drive training package, prepared English tokenizer/dataset and latest verified checkpoint.

## Start here
1. In Colab, select **Runtime → Change runtime type → None / CPU** (the CPU-only option), then Save. No GPU is needed or requested.
2. Run the single code cell and approve Google Drive access. If the Google authorization page says **“Please close this tab / window”**, close only that authorization tab and return to Colab.
3. Confirm the output says `CPU-only mode confirmed` and shows the last verified training step. The notebook should then resume from that checkpoint.

## CPU settings and checkpoint safety
- CPU-only is enforced with `CUDA_VISIBLE_DEVICES=''` before importing PyTorch. The notebook stops if CUDA remains visible instead of silently switching to GPU.
- Uses at most four CPU threads (based on the runtime's available cores) and avoids CUDA/FP16 setup.
- Keeps batch size 4 × gradient accumulation 2 = effective batch size 8, matching the existing training contract and 4,096 tokens per optimizer step.
- Keeps the 512-token context, 4,096-token vocabulary, English TinyStories corpus and 20,000-step target.
- Reuses verified Drive tokenizer/data/checkpoints. Checkpoints are still verified every 10 optimizer steps.

**Important:** CPU training will be substantially slower than the Tesla T4. Actual speed depends on Colab's allocated CPU cores. This notebook prioritizes continuing safely from your saved progress, not claiming GPU speed. Do not run another Q-Mini trainer at the same time.

**Scope:** English TinyStories base pretraining, not yet a finished general-purpose conversational assistant.


In [ ]:
# Q-MINI ENGLISH 512 — ONE-CELL AUTO PRETRAINER
# Release marker: confirms that this fresh notebook, not an old Colab tab, is running.
print("NOTEBOOK BUILD: QMINI-ENGLISH-512-CPU-ONLY-RESUME-V9-2026-11-15")
# Uses the existing verified initial model. Does not rebuild random weights.
# Downloads a pinned, license-declared TinyStories English corpus; trains and saves
# resumable checkpoints and the final Q2-v2 model to Google Drive.

# CPU-only lock: hide all GPUs BEFORE importing PyTorch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = str(min(4, max(1, os.cpu_count() or 2)))
os.environ["MKL_NUM_THREADS"] = os.environ["OMP_NUM_THREADS"]
os.environ["OPENBLAS_NUM_THREADS"] = os.environ["OMP_NUM_THREADS"]

from pathlib import Path
import torch

CPU_THREADS = min(4, max(1, os.cpu_count() or 2))
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

if torch.cuda.is_available():
    raise RuntimeError(
        "CPU-only lock did not take effect. Restart the Colab runtime and run this notebook again. "
        "No training or checkpoint files have been touched."
    )
print("CPU-only mode confirmed.")
print("CUDA_VISIBLE_DEVICES:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("PyTorch:", torch.__version__)
print("CPU threads:", torch.get_num_threads())
print("No GPU will be used by this notebook.")

from google.colab import drive
print("\nGoogle Drive authorization: approve access in the Google tab.")
print("When the page says 'Please close this tab / window', close only that authorization tab and return here.")
try:
    drive.mount("/content/drive", force_remount=False, timeout_ms=600000)
except Exception as drive_error:
    raise RuntimeError(
        "Google Drive did not mount. If the authorization tab showed 'Please close this tab / window', "
        "close that tab (not the Colab notebook) and return here. If this error repeats, share the full "
        "red traceback below; the training checkpoint and prepared data have not been touched."
    ) from drive_error
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive mount returned but MyDrive was not found; training will not start.")
print("Google Drive mount verified: /content/drive/MyDrive")

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid

# -------------------------- Fixed, verified settings --------------------------
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
EXPECTED_INITIAL_MODEL_REVISION = "b262787a81dd8297bd970624a55cff44b4424fe9"
TRAINING_SOURCE_REVISION = "e6e6aefe0b303013c817a110fd46b38613928360"
MODEL_CONFIG_REL = "configs/qmini_light_0.7m.json"

DATASET_ID = "roneneldan/TinyStories"
DATASET_REVISION = "6d02f92396f7e3e60c2251070b6e0b168568074b"
DATASET_SUBSET = "data"
DATASET_PAGE = (
    "https://huggingface.co/datasets/roneneldan/TinyStories/tree/"
    + DATASET_REVISION + "/" + DATASET_SUBSET
)
DATASET_FILES = [
    {
        "name": "train-00000-of-00004-2d5a1467fff1081b.parquet",
        "url": "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/" + DATASET_REVISION + "/data/train-00000-of-00004-2d5a1467fff1081b.parquet",
        "sha256": "77cf780cebe52b6e83e3a2ac84bc56d8059363113e41d17a023f1d8b2ed0fc0b",
    },
    {
        "name": "train-00001-of-00004-5852b56a2bd28fd9.parquet",
        "url": "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/" + DATASET_REVISION + "/data/train-00001-of-00004-5852b56a2bd28fd9.parquet",
        "sha256": "e2f173a6dcc2b5895183e8ad376e115bd5c06764eb0cfc01ce1add156b2d017f",
    },
    {
        "name": "train-00002-of-00004-a26307300439e943.parquet",
        "url": "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/" + DATASET_REVISION + "/data/train-00002-of-00004-a26307300439e943.parquet",
        "sha256": "cac73cd1a0973a7fccde022a5a5c78f8ba58ea6849541835c58e7c5fc747f2ef",
    },
    {
        "name": "train-00003-of-00004-d243063613e5a057.parquet",
        "url": "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/" + DATASET_REVISION + "/data/train-00003-of-00004-d243063613e5a057.parquet",
        "sha256": "862514ae75723ddd339337f1ef7617b88f82290dcddd57445e1d38d257ca7891",
    },
]
VOCAB_SIZE = 4096
VALIDATION_FRACTION = 0.02
TRAIN_SEQUENCE_LENGTH = 512
TARGET_OPTIMIZER_STEPS = 20000
BATCH_SIZE = 4
GRAD_ACCUM = 2
GLOBAL_BATCH_SIZE = 8
if BATCH_SIZE * GRAD_ACCUM != GLOBAL_BATCH_SIZE:
    raise RuntimeError("The optimized microbatch must preserve effective batch size 8.")
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY = 10
EVALUATE_EVERY = 100
EVALUATION_BATCHES = 8
CPU_THREAD_COUNT = CPU_THREADS
LEASE_STALE_SECONDS = 180
LEASE_HEARTBEAT_SECONDS = 45

def run_checked(command, *, cwd=None, capture_output=False, env=None):
    command = [str(item) for item in command]
    print("$", " ".join(command))
    # Always retain child output: notebook tracebacks otherwise hide the real
    # tokenizer/data exception behind a generic CalledProcessError.
    result = subprocess.run(
        command, cwd=str(cwd) if cwd else None, check=False,
        text=True, capture_output=True, env=env,
    )
    if result.stdout:
        print(result.stdout[-16000:])
    if result.returncode != 0:
        if result.stderr:
            print("----- subprocess stderr -----")
            print(result.stderr[-24000:])
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: "
            + " ".join(command)
            + "\\nSee the captured stdout/stderr immediately above."
        )
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

def download_verified(record, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = record["sha256"]
    if destination.is_file() and sha256_file(destination) == expected:
        print("Already downloaded and verified:", destination.name)
        return
    if destination.exists():
        destination.unlink()
    partial = destination.with_name(destination.name + ".part")
    partial.unlink(missing_ok=True)
    print("Downloading:", record["name"])
    request = Request(
        record["url"],
        headers={"User-Agent": "Q-Mini-Research-Training/1.0"},
    )
    total = 0
    with urlopen(request, timeout=90) as response, partial.open("wb") as output:
        while True:
            chunk = response.read(8 * 1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            total += len(chunk)
            if total and total % (32 * 1024 * 1024) < len(chunk):
                print("  received", f"{total / (1024**2):.1f} MiB")
    actual = sha256_file(partial)
    if actual != expected:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f"Dataset SHA-256 mismatch for {record['name']}; refusing to train on an unverified file."
        )
    partial.replace(destination)
    print("Verified:", destination.name, f"({total / (1024**2):.1f} MiB)")

print("\n[1/8] Checking CPU-only runtime...")
MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not connected. Approve the Drive prompt, return to Colab, then rerun.")

from packaging.version import Version
torch_public_version = torch.__version__.split("+", 1)[0]
if Version(torch_public_version) < Version("2.10.0"):
    raise RuntimeError(
        f"Q-Mini needs PyTorch >=2.10.0; this runtime has {torch.__version__}. "
        "Use a current Colab CPU runtime and rerun."
    )
if torch.cuda.is_available():
    raise RuntimeError("CPU-only safety check failed: CUDA became visible. No training was started.")
print("CPU-only preflight PASSED.")
print("PyTorch:", torch.__version__)
print("CPU thread count:", torch.get_num_threads())
try:
    import psutil
    _vm = psutil.virtual_memory()
    print(f"System RAM: {_vm.available / (1024**3):.2f} GiB available / {_vm.total / (1024**3):.2f} GiB total")
except Exception:
    print("System RAM monitor unavailable; continuing with the built-in memory preflight.")

# Preserve the Colab-provided PyTorch installation.
run_checked([
    sys.executable, "-m", "pip", "install", "--quiet",
    "numpy>=1.24", "tokenizers>=0.15", "pyarrow>=15.0", "packaging>=23",
])

print("\n[2/8] Checking Google Drive for a compatible initial model...")
INITIAL_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "initial_brains"
INITIAL_ROOT.mkdir(parents=True, exist_ok=True)
valid_initials = []
for folder in INITIAL_ROOT.glob("QMini-qmini_light_0.7m-Q2v2-initial-*"):
    manifest_path = folder / "model_manifest.json"
    config_path = folder / "config.json"
    model_path = folder / "model.qmini.npz"
    if not (manifest_path.is_file() and config_path.is_file() and model_path.is_file()):
        continue
    try:
        candidate_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        candidate_config = json.loads(config_path.read_text(encoding="utf-8"))
        if (
            candidate_manifest.get("format") == "qmini-initial-brain-v1"
            and candidate_manifest.get("status") == "untrained_initialization"
            and candidate_manifest.get("training_steps") == 0
            and candidate_manifest.get("optimizer_steps") == 0
            and candidate_manifest.get("source_revision") == EXPECTED_INITIAL_MODEL_REVISION
            and candidate_manifest.get("parameters") == 705742
            and candidate_config.get("max_context") == 512
            and candidate_config.get("vocab_size") == VOCAB_SIZE
            and candidate_manifest.get("packed_export_sha256") == sha256_file(model_path)
        ):
            valid_initials.append((folder.stat().st_mtime, folder, candidate_manifest))
    except (OSError, ValueError, TypeError):
        continue

SOURCE_REVISION = TRAINING_SOURCE_REVISION

print("\n[3/8] Fetching the exact Q-Mini source revision...")
REPO_DIR = Path("/content") / ("QMini-training-source-" + uuid.uuid4().hex[:8])
REPO_DIR.mkdir(parents=True, exist_ok=False)
run_checked(["git", "init"], cwd=REPO_DIR)
run_checked(["git", "remote", "add", "origin", REPOSITORY_URL], cwd=REPO_DIR)
run_checked(["git", "fetch", "--depth", "1", "origin", SOURCE_REVISION], cwd=REPO_DIR)
run_checked(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
actual_revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("Downloaded Q-Mini source revision does not match the initial model manifest.")
if not (REPO_DIR / "qmini" / "cli.py").is_file():
    raise FileNotFoundError("The pinned Q-Mini source checkout is incomplete.")
sys.path.insert(0, str(REPO_DIR))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

# This kernel may have run an older notebook already. Remove cached Q-Mini
# modules so imports below definitely come from the newly pinned, fixed checkout.
for _loaded_module in tuple(sys.modules):
    if _loaded_module == "qmini" or _loaded_module.startswith("qmini."):
        del sys.modules[_loaded_module]


from qmini.config import QMiniConfig
from qmini.export import verify_packed_export
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS
from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
)

print("\\n[3.5/8] Running the SQLite thread-migration preflight before dataset work...")
from concurrent.futures import ThreadPoolExecutor
from qmini.data import iter_training_text_records

_preflight_dir = Path("/content") / ("qmini-thread-hop-" + uuid.uuid4().hex[:8])
_preflight_dir.mkdir(parents=True, exist_ok=False)
_preflight_corpus = _preflight_dir / "thread-hop.txt"
_preflight_iterator = None
try:
    _preflight_corpus.write_text(
        "\n\n".join(
            f"independent Q-Mini preflight document {index} contains enough unique words"
            for index in range(48)
        ),
        encoding="utf-8",
    )
    _preflight_iterator = iter_training_text_records(
        [_preflight_corpus], valid_fraction=0.02, near_duplicate_distance=0
    )
    _preflight_first = next(_preflight_iterator)
    with ThreadPoolExecutor(max_workers=1) as _executor:
        _preflight_second = _executor.submit(next, _preflight_iterator).result()
    _preflight_remaining = list(_preflight_iterator)
    if len({_preflight_first, _preflight_second, *_preflight_remaining}) < 2:
        raise RuntimeError("SQLite thread-migration preflight did not yield unique training records.")
    print(
        "SQLite thread-migration preflight PASSED:",
        2 + len(_preflight_remaining), "unique training records read across threads.",
    )
finally:
    if _preflight_iterator is not None:
        _preflight_iterator.close()
    shutil.rmtree(_preflight_dir, ignore_errors=True)

base_config = QMiniConfig.from_json(REPO_DIR / MODEL_CONFIG_REL)
if not valid_initials:
    print("No compatible initial brain found; building and verifying the step-zero Q2-v2 model now...")
    _initial_config = base_config
    if (
        _initial_config.max_context != TRAIN_SEQUENCE_LENGTH
        or _initial_config.vocab_size != VOCAB_SIZE
        or _initial_config.estimated_parameter_count() != 705742
    ):
        raise RuntimeError("Pinned Q-Mini config does not match the expected 705,742-parameter / 512-token English model.")
    _new_initial_dir = INITIAL_ROOT / (
        "QMini-qmini_light_0.7m-Q2v2-initial-"
        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + "-" + uuid.uuid4().hex[:8]
    )
    from qmini.initial_build import build_initial_brain
    _new_initial_manifest = build_initial_brain(
        _initial_config, _new_initial_dir,
        source_revision=EXPECTED_INITIAL_MODEL_REVISION,
    )
    valid_initials.append((_new_initial_dir.stat().st_mtime, _new_initial_dir, _new_initial_manifest))

valid_initials.sort(key=lambda item: item[0], reverse=True)
_initial_time, INITIAL_DIR_SOURCE, INITIAL_MANIFEST = valid_initials[0]
INITIAL_MODEL_SOURCE = INITIAL_DIR_SOURCE / "model.qmini.npz"
INITIAL_SHA256 = INITIAL_MANIFEST["packed_export_sha256"]
INITIAL_MODEL_SOURCE_REVISION = INITIAL_MANIFEST.get("source_revision")
if INITIAL_MODEL_SOURCE_REVISION != EXPECTED_INITIAL_MODEL_REVISION:
    raise RuntimeError(
        "The saved initial model was not created from the pinned architecture revision. "
        f"Model revision={INITIAL_MODEL_SOURCE_REVISION!r}; "
        f"expected={EXPECTED_INITIAL_MODEL_REVISION!r}."
    )
print("Selected initial brain:", INITIAL_DIR_SOURCE.name)
print("Initial model SHA-256:", INITIAL_SHA256)


if base_config.max_context != TRAIN_SEQUENCE_LENGTH:
    raise RuntimeError("The GitHub model config does not have the required 512-token context.")
if base_config.estimated_parameter_count() != INITIAL_MANIFEST["parameters"]:
    raise RuntimeError("GitHub model config and saved initial model parameter count differ.")
initial_report = verify_packed_export(INITIAL_MODEL_SOURCE)
if (
    initial_report.get("format") != "qmini-packed-v2"
    or initial_report.get("q2_version") != 2
    or initial_report.get("parameter_count") != base_config.estimated_parameter_count()
):
    raise RuntimeError("The existing model failed Q2-v2 verification.")

print("\n[4/8] Downloading and verifying the TinyStories English corpus...")
DATA_DIR = MY_DRIVE / "QMini-Light-0.7M" / "datasets" / "english_tinystories_v1"
DATA_DIR.mkdir(parents=True, exist_ok=True)
for record in DATASET_FILES:
    download_verified(record, DATA_DIR / record["name"])

SOURCE_METADATA_PATH = (
    MY_DRIVE / "QMini-Light-0.7M" / "source_metadata"
    / "qmini-english-tinystories-v1.json"
)
SOURCE_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if SOURCE_METADATA_PATH.is_file():
    source_metadata = json.loads(SOURCE_METADATA_PATH.read_text(encoding="utf-8"))
    _expected_sources = [
        {
            "path": record["name"],
            "source_name": "roneneldan/TinyStories English training split",
            "source_uri": DATASET_PAGE,
            "license_spdx": "CDLA-Sharing-1.0",
            "language": "en",
            "transformation_version": "qmini-parquet-source-v1",
        }
        for record in DATASET_FILES
    ]
    _saved_sources = [
        {key: source.get(key) for key in _expected_sources[0]}
        for source in source_metadata.get("sources", [])
    ]
    if (
        source_metadata.get("format") != "qmini-source-provenance-v1"
        or _saved_sources != _expected_sources
    ):
        raise RuntimeError("Saved Drive source-provenance metadata does not match the pinned English corpus.")
    print("Reusing the original Drive source-provenance record.")
else:
    source_metadata = {
        "format": "qmini-source-provenance-v1",
        "sources": [
            {
                "path": record["name"],
                "source_name": "roneneldan/TinyStories English training split",
                "source_uri": DATASET_PAGE,
                "license_spdx": "CDLA-Sharing-1.0",
                "collected_at": datetime.now(timezone.utc).date().isoformat(),
                "language": "en",
                "transformation_version": "qmini-parquet-source-v1",
            }
            for record in DATASET_FILES
        ],
    }
    write_json(SOURCE_METADATA_PATH, source_metadata)
data_fingerprints = [
    {"name": item["name"], "sha256": sha256_file(DATA_DIR / item["name"])}
    for item in DATASET_FILES
]
package_identity = {
    "format": "qmini-english-training-package-v1",
    # Keep the data/model package key stable across training-code-only fixes.
    # The initial architecture/weights remain pinned to the original revision.
    "source_revision": EXPECTED_INITIAL_MODEL_REVISION,
    "initial_model_sha256": INITIAL_SHA256,
    "dataset": DATASET_PAGE,
    "dataset_files": data_fingerprints,
    "vocab_size": VOCAB_SIZE,
    "context_length": TRAIN_SEQUENCE_LENGTH,
    "validation_fraction": VALIDATION_FRACTION,
    "target_optimizer_steps": TARGET_OPTIMIZER_STEPS,
}
PACKAGE_KEY = hashlib.sha256(
    json.dumps(package_identity, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
PACKAGE_NAME = "QMini-English-512-Q2v2-" + PACKAGE_KEY
PACKAGE_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "training_ready"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_DIR = PACKAGE_ROOT / PACKAGE_NAME
STAGE_DIR = PACKAGE_ROOT / (".resume-" + PACKAGE_NAME)

print("Corpus files:", len(data_fingerprints))
print("Corpus size:", f"{sum((DATA_DIR / r['name']).stat().st_size for r in DATASET_FILES) / (1024**2):.1f} MiB")
print("Training package:", PACKAGE_DIR)

def verify_training_package(package_dir):
    package_dir = Path(package_dir)
    initial_dir = package_dir / "initial_brain"
    tokenizer_path = package_dir / "tokenizer" / "tokenizer.json"
    dataset_dir = package_dir / "dataset"
    runtime_dir = package_dir / "training_runtime"
    runtime_manifest_path = runtime_dir / "TRAINING_RUNTIME.json"
    config_path = initial_dir / "config.json"
    model_path = initial_dir / "model.qmini.npz"
    manifest_path = initial_dir / "model_manifest.json"
    progress_path = package_dir / "training_state" / "training_progress.json"
    required = [
        config_path, model_path, manifest_path, tokenizer_path,
        Path(str(tokenizer_path) + ".qmini.json"),
        package_dir / "active_config.json",
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        dataset_dir / "train.mask.bin", dataset_dir / "valid.mask.bin",
        dataset_dir / "dataset.json",
        runtime_manifest_path, runtime_dir / "qmini" / "train.py",
        runtime_dir / "qmini" / "checkpointing.py",
        progress_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Training package is incomplete: " + ", ".join(missing))

    config = QMiniConfig.from_json(config_path)
    active_config = QMiniConfig.from_json(package_dir / "active_config.json")
    if config.to_dict() != active_config.to_dict():
        raise RuntimeError("Tokenizer changed the model architecture or vocabulary unexpectedly.")
    if config.max_context != 512 or config.vocab_size != VOCAB_SIZE:
        raise RuntimeError("Training package does not preserve 512 context and 4096 vocabulary.")
    if config.estimated_parameter_count() != 705742:
        raise RuntimeError("Training package parameter count changed unexpectedly.")
    loaded_tokenizer = load_tokenizer(tokenizer_path)
    if loaded_tokenizer.get_vocab_size() != VOCAB_SIZE:
        raise RuntimeError("Tokenizer vocabulary does not match the initial model.")
    for token in SPECIAL_TOKENS:
        if loaded_tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Required structural token is missing: {token}")

    _verify_initial_brain_artifact(model_path, config)
    _verify_prepared_dataset_manifest(
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        tokenizer_path, dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", config.vocab_size,
    )
    train_array = _load_tokens(dataset_dir / "train.bin", "Training")
    valid_array = _load_tokens(dataset_dir / "valid.bin", "Validation")
    _validate_token_ids(train_array, config.vocab_size, "Training")
    _validate_token_ids(valid_array, config.vocab_size, "Validation")
    if len(train_array) < 514 or len(valid_array) < 514:
        raise RuntimeError("Prepared data is too small for 512-token training sequences.")
    if (dataset_dir / "train.mask.bin").stat().st_size != (dataset_dir / "train.bin").stat().st_size // 2:
        raise RuntimeError("Training mask size does not match the token stream.")
    if (dataset_dir / "valid.mask.bin").stat().st_size != (dataset_dir / "valid.bin").stat().st_size // 2:
        raise RuntimeError("Validation mask size does not match the token stream.")

    runtime_manifest = json.loads(runtime_manifest_path.read_text(encoding="utf-8"))
    if runtime_manifest.get("enabled") is not False or runtime_manifest.get("requires_manual_activation") is not True:
        raise RuntimeError("Training runtime was unexpectedly enabled in the package.")
    for record in runtime_manifest.get("source_files", []):
        relative = Path(record["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Unsafe path in training runtime manifest.")
        source = runtime_dir / relative
        if not source.is_file() or sha256_file(source) != record["sha256"]:
            raise RuntimeError(f"Training source hash mismatch: {relative}")
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if model_manifest.get("training_runtime_manifest_sha256") != sha256_file(runtime_manifest_path):
        raise RuntimeError("Training runtime manifest hash does not match model manifest.")
    if model_manifest.get("packed_export_sha256") != sha256_file(model_path):
        raise RuntimeError("Initial model fingerprint changed inside the training package.")
    return config, model_manifest, train_array, valid_array

if PACKAGE_DIR.is_dir():
    print("\nExisting training package found; verifying and reusing it.")
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)

    # Refresh the disabled, hash-checked source snapshot to the fixed training
    # revision while retaining the already prepared tokenizer and dataset.
    RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
    shutil.rmtree(RUNTIME_DIR / "qmini", ignore_errors=True)
    shutil.copytree(
        REPO_DIR / "qmini",
        RUNTIME_DIR / "qmini",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
    )
    shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR / "pyproject.toml")
    _runtime_files = []
    for _source in sorted(
        p for p in RUNTIME_DIR.rglob("*")
        if p.is_file() and p.name != "TRAINING_RUNTIME.json"
    ):
        _runtime_files.append({
            "path": _source.relative_to(RUNTIME_DIR).as_posix(),
            "sha256": sha256_file(_source),
            "size_bytes": _source.stat().st_size,
        })
    write_json(RUNTIME_DIR / "TRAINING_RUNTIME.json", {
        "format": "qmini-training-runtime-v1",
        "enabled": False,
        "requires_manual_activation": True,
        "entrypoint": "qmini.train.train_model",
        "checkpoint_backend": "qmini.checkpointing",
        "source_revision": SOURCE_REVISION,
        "source_files": _runtime_files,
    })
    _initial_manifest_path = PACKAGE_DIR / "initial_brain" / "model_manifest.json"
    _initial_manifest = json.loads(_initial_manifest_path.read_text(encoding="utf-8"))
    _initial_manifest["training_runtime_manifest_sha256"] = sha256_file(
        RUNTIME_DIR / "TRAINING_RUNTIME.json"
    )
    write_json(_initial_manifest_path, _initial_manifest)

    _project_source = PACKAGE_DIR / "project_source"
    shutil.rmtree(_project_source, ignore_errors=True)
    shutil.copytree(
        REPO_DIR, _project_source,
        ignore=shutil.ignore_patterns(
            ".git", "__pycache__", "*.pyc", ".pytest_cache",
            ".venv", "venv", ".mypy_cache", ".ruff_cache",
        ),
    )
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
    print("Updated training-code snapshot; original tokenizer and prepared dataset were preserved.")
else:
    print("\n[5/8] Reusing completed Drive stages or building only the missing stage...")
    # This staging directory persists across Colab runtime restarts.
    LOCAL_PACKAGE = STAGE_DIR
    LOCAL_PACKAGE.mkdir(parents=True, exist_ok=True)
    for _subdir in ("tokenizer", "dataset", "initial_brain", "training_state"):
        (LOCAL_PACKAGE / _subdir).mkdir(parents=True, exist_ok=True)

    TOKENIZER_PATH = LOCAL_PACKAGE / "tokenizer" / "tokenizer.json"
    ACTIVE_CONFIG_PATH = LOCAL_PACKAGE / "active_config.json"
    STAGED_DATASET_DIR = LOCAL_PACKAGE / "dataset"
    # Keep the large prepare operation on Colab's fast local disk; copy its
    # completed outputs to Drive immediately when the stage finishes.
    DATASET_OUT_DIR = Path("/content") / ("qmini-dataset-" + PACKAGE_KEY)

    # Recover completed artifacts from the previous notebook's /content staging
    # directory when the same Colab runtime is still alive. This specifically
    # preserves the tokenizer already built by the earlier non-resumable notebook.
    for _old_stage in sorted(Path("/content").glob(PACKAGE_NAME + "-staging-*")):
        _old_tokenizer = _old_stage / "tokenizer" / "tokenizer.json"
        _old_sidecar = Path(str(_old_tokenizer) + ".qmini.json")
        if (
            _old_tokenizer.is_file() and _old_sidecar.is_file()
            and not TOKENIZER_PATH.is_file()
        ):
            shutil.copy2(_old_tokenizer, TOKENIZER_PATH)
            shutil.copy2(_old_sidecar, Path(str(TOKENIZER_PATH) + ".qmini.json"))
            _old_active = _old_stage / "active_config.json"
            if _old_active.is_file():
                shutil.copy2(_old_active, ACTIVE_CONFIG_PATH)
            print("Recovered the previously completed tokenizer from this Colab runtime.")

        _old_dataset_files = (
            "train.bin", "valid.bin", "train.mask.bin", "valid.mask.bin", "dataset.json"
        )
        if (
            all((_old_stage / "dataset" / name).is_file() for name in _old_dataset_files)
            and not any((STAGED_DATASET_DIR / name).is_file() for name in _old_dataset_files)
        ):
            _dataset_recovery = LOCAL_PACKAGE / (".dataset-recovery-" + uuid.uuid4().hex[:8])
            shutil.copytree(_old_stage / "dataset", _dataset_recovery)
            shutil.rmtree(STAGED_DATASET_DIR, ignore_errors=True)
            _dataset_recovery.rename(STAGED_DATASET_DIR)
            print("Recovered the completed train/validation dataset from this Colab runtime.")


    # Inspect the downloaded Parquet schema and a few records before tokenizer
    # fitting, so a dataset-layout problem is reported clearly and early.
    import pyarrow.parquet as pq
    for _record in DATASET_FILES:
        _parquet_path = DATA_DIR / _record["name"]
        _parquet = pq.ParquetFile(_parquet_path)
        print(
            "Parquet check:", _record["name"],
            "| rows:", _parquet.metadata.num_rows,
            "| columns:", _parquet.schema.names,
        )
        if "text" not in _parquet.schema.names:
            raise RuntimeError(
                f"Expected a 'text' column in {_record['name']}; "
                f"found {_parquet.schema.names}. Refusing to train on an unknown schema."
            )
        _sample = next(_parquet.iter_batches(batch_size=1), None)
        if _sample is None or _sample.num_rows == 0:
            raise RuntimeError(f"Dataset shard is empty: {_record['name']}")
        _sample_text = _sample.column(_sample.schema.get_field_index("text"))[0].as_py()
        if not isinstance(_sample_text, str) or not _sample_text.strip():
            raise RuntimeError(
                f"The 'text' column in {_record['name']} does not contain non-empty text."
            )
        print("  First record text length:", len(_sample_text))

    # Reuse a completed tokenizer only after checking its manifest, SHA-256,
    # vocabulary, required control tokens and model config.
    from qmini.tokenizer import train_tokenizer
    tokenizer_sidecar = Path(str(TOKENIZER_PATH) + ".qmini.json")
    tokenizer_cache_ok = False
    if TOKENIZER_PATH.is_file() and tokenizer_sidecar.is_file():
        try:
            _cached_tokenizer = load_tokenizer(TOKENIZER_PATH)
            _cached_meta = json.loads(tokenizer_sidecar.read_text(encoding="utf-8"))
            _cached_config = (
                QMiniConfig.from_json(ACTIVE_CONFIG_PATH)
                if ACTIVE_CONFIG_PATH.is_file() else base_config
            )
            tokenizer_cache_ok = (
                _cached_meta.get("format") == "qmini-tokenizer-manifest-v1"
                and _cached_meta.get("tokenizer_sha256") == sha256_file(TOKENIZER_PATH)
                and _cached_meta.get("validation_fraction") == VALIDATION_FRACTION
                and _cached_meta.get("near_duplicate_hamming_distance") == 0
                and _cached_tokenizer.get_vocab_size() == VOCAB_SIZE
                and all(_cached_tokenizer.token_to_id(token) is not None for token in SPECIAL_TOKENS)
                and _cached_config.to_dict() == base_config.to_dict()
            )
        except Exception as _cache_error:
            print("Saved tokenizer did not pass verification; rebuilding this stage:", repr(_cache_error))
            tokenizer_cache_ok = False

    if tokenizer_cache_ok:
        trained_tokenizer = _cached_tokenizer
        active_config = _cached_config
        if not ACTIVE_CONFIG_PATH.is_file():
            active_config.save(ACTIVE_CONFIG_PATH)
        print("Reusing verified English tokenizer from Google Drive; tokenizer training skipped.")
    else:
        TOKENIZER_PATH.unlink(missing_ok=True)
        tokenizer_sidecar.unlink(missing_ok=True)
        ACTIVE_CONFIG_PATH.unlink(missing_ok=True)
        print("Tokenizer implementation: direct in-process training (V4 thread-safe data pipeline)")
        try:
            trained_tokenizer = train_tokenizer(
                [DATA_DIR],
                TOKENIZER_PATH,
                vocab_size=VOCAB_SIZE,
                min_frequency=2,
                overwrite=True,
                validation_fraction=VALIDATION_FRACTION,
                near_duplicate_distance=0,
                source_metadata_path=SOURCE_METADATA_PATH,
            )
            active_config = QMiniConfig.from_json(REPO_DIR / MODEL_CONFIG_REL)
            active_config.vocab_size = trained_tokenizer.get_vocab_size()
            active_config.validate()
            active_config.save(ACTIVE_CONFIG_PATH)
            print(
                "Tokenizer completed and saved to Google Drive:",
                "| vocab:", trained_tokenizer.get_vocab_size(),
                "| output:", TOKENIZER_PATH,
            )
        except Exception:
            import traceback
            print("TOKENIZER STAGE FAILED — full original Python traceback:")
            traceback.print_exc()
            raise

    active_config = QMiniConfig.from_json(ACTIVE_CONFIG_PATH)
    if active_config.to_dict() != base_config.to_dict():
        raise RuntimeError(
            "Tokenizer vocabulary/config differs from the saved initial model. "
            "No weights will be changed; inspect the corpus/tokenizer before continuing."
        )

    # A completed train/validation dataset is also cached on Drive. The current
    # prepare command uses two full passes and does not checkpoint mid-command:
    # interruption during prepare reruns only prepare, while keeping the tokenizer.
    dataset_cache_ok = False
    if all((STAGED_DATASET_DIR / name).is_file() for name in (
        "train.bin", "valid.bin", "train.mask.bin", "valid.mask.bin", "dataset.json"
    )):
        try:
            _verify_prepared_dataset_manifest(
                STAGED_DATASET_DIR / "train.bin",
                STAGED_DATASET_DIR / "valid.bin",
                TOKENIZER_PATH,
                STAGED_DATASET_DIR / "train.mask.bin",
                STAGED_DATASET_DIR / "valid.mask.bin",
                VOCAB_SIZE,
            )
            _cached_train = _load_tokens(STAGED_DATASET_DIR / "train.bin", "Cached training")
            _cached_valid = _load_tokens(STAGED_DATASET_DIR / "valid.bin", "Cached validation")
            _validate_token_ids(_cached_train, VOCAB_SIZE, "Cached training")
            _validate_token_ids(_cached_valid, VOCAB_SIZE, "Cached validation")
            dataset_cache_ok = len(_cached_train) >= 514 and len(_cached_valid) >= 514
        except Exception as _dataset_cache_error:
            print("Prepared dataset cache did not pass verification; rebuilding this stage:", repr(_dataset_cache_error))
            dataset_cache_ok = False

    if dataset_cache_ok:
        print("Reusing verified train/validation token streams from Google Drive; prepare skipped.")
    else:
        shutil.rmtree(STAGED_DATASET_DIR, ignore_errors=True)
        shutil.rmtree(DATASET_OUT_DIR, ignore_errors=True)
        DATASET_OUT_DIR.mkdir(parents=True, exist_ok=False)
        print("Preparing train/validation streams. The completed tokenizer remains safely cached on Drive.")
        run_checked([
            sys.executable, "-m", "qmini.cli", "prepare",
            "--data-dir", DATA_DIR,
            "--tokenizer", TOKENIZER_PATH,
            "--output-dir", DATASET_OUT_DIR,
            "--valid-fraction", str(VALIDATION_FRACTION),
            "--near-duplicate-hamming-distance", "0",
            "--loss-mask", "all",
            "--source-metadata", SOURCE_METADATA_PATH,
        ], cwd=REPO_DIR)
        _dataset_tmp = LOCAL_PACKAGE / (".dataset-copy-" + uuid.uuid4().hex[:8])
        shutil.copytree(DATASET_OUT_DIR, _dataset_tmp)
        _dataset_tmp.rename(STAGED_DATASET_DIR)
        _verify_prepared_dataset_manifest(
            STAGED_DATASET_DIR / "train.bin",
            STAGED_DATASET_DIR / "valid.bin",
            TOKENIZER_PATH,
            STAGED_DATASET_DIR / "train.mask.bin",
            STAGED_DATASET_DIR / "valid.mask.bin",
            VOCAB_SIZE,
        )
        print("Prepared dataset copied to Drive and verified.")

    for _stale_dataset in LOCAL_PACKAGE.glob(".dataset-copy-*"):
        if _stale_dataset.is_dir():
            shutil.rmtree(_stale_dataset, ignore_errors=True)
    for _stale_dataset in LOCAL_PACKAGE.glob(".dataset-recovery-*"):
        if _stale_dataset.is_dir():
            shutil.rmtree(_stale_dataset, ignore_errors=True)

    # Reuse the exact initial artifact from the previous successful build.
    for filename in ("config.json", "model.qmini.npz", "model_manifest.json"):
        _dest = LOCAL_PACKAGE / "initial_brain" / filename
        if not _dest.is_file():
            shutil.copy2(INITIAL_DIR_SOURCE / filename, _dest)

    # Keep the complete source checkout and a hash-checked training-runtime bundle.
    _project_copy = LOCAL_PACKAGE / "project_source"
    if not (_project_copy / "qmini" / "train.py").is_file():
        shutil.rmtree(_project_copy, ignore_errors=True)
        shutil.copytree(
            REPO_DIR, _project_copy,
            ignore=shutil.ignore_patterns(
                ".git", "__pycache__", "*.pyc", ".pytest_cache",
                ".venv", "venv", ".mypy_cache", ".ruff_cache",
            ),
        )
    RUNTIME_DIR_LOCAL = LOCAL_PACKAGE / "training_runtime"
    if not (RUNTIME_DIR_LOCAL / "TRAINING_RUNTIME.json").is_file():
        shutil.rmtree(RUNTIME_DIR_LOCAL, ignore_errors=True)
        RUNTIME_DIR_LOCAL.mkdir(parents=True)
        shutil.copytree(
            REPO_DIR / "qmini", RUNTIME_DIR_LOCAL / "qmini",
            ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
        )
        shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR_LOCAL / "pyproject.toml")
    write_json(RUNTIME_DIR_LOCAL / "TRAINING_DEFAULTS.json", {
        "format": "qmini-training-defaults-v1",
        "enabled": False,
        "sequence_length": TRAIN_SEQUENCE_LENGTH,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "checkpoint_every_optimizer_steps": CHECKPOINT_EVERY,
        "evaluation_every_optimizer_steps": EVALUATE_EVERY,
        "total_optimizer_steps": TARGET_OPTIMIZER_STEPS,
        "compute_device": "cpu",

        "cpu_threads": CPU_THREAD_COUNT,
    })
    (RUNTIME_DIR_LOCAL / "README.md").write_text(
        "# Q-Mini English training runtime\n\n"
        "Exact qmini source matching the step-zero initial model source revision. "
        "Training settings are recorded in TRAINING_DEFAULTS.json. Training is "
        "activated only by the one-click training notebook.\n",
        encoding="utf-8",
    )

    runtime_files = []
    for source in sorted(p for p in RUNTIME_DIR_LOCAL.rglob("*") if p.is_file() and p.name != "TRAINING_RUNTIME.json"):
        runtime_files.append({
            "path": source.relative_to(RUNTIME_DIR_LOCAL).as_posix(),
            "sha256": sha256_file(source),
            "size_bytes": source.stat().st_size,
        })
    runtime_manifest = {
        "format": "qmini-training-runtime-v1",
        "enabled": False,
        "requires_manual_activation": True,
        "entrypoint": "qmini.train.train_model",
        "checkpoint_backend": "qmini.checkpointing",
        "source_revision": SOURCE_REVISION,
        "source_files": runtime_files,
    }
    write_json(RUNTIME_DIR_LOCAL / "TRAINING_RUNTIME.json", runtime_manifest)

    write_json(LOCAL_PACKAGE / "training_state" / "training_progress.json", {
        "format": "qmini-training-progress-v1",
        "status": "not_started",
        "current_step": 0,
        "target_steps": TARGET_OPTIMIZER_STEPS,
        "progress_percent": 0.0,
        "tokens_seen": 0,
        "updated_at": datetime.now(timezone.utc).isoformat(),
    })

    # Bind the existing initial weights to this exact tokenizer/data/runtime.
    manifest_path = LOCAL_PACKAGE / "initial_brain" / "model_manifest.json"
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    dataset_manifest_path = STAGED_DATASET_DIR / "dataset.json"
    dataset_manifest = json.loads(dataset_manifest_path.read_text(encoding="utf-8"))
    tokenizer_sidecar = Path(str(TOKENIZER_PATH) + ".qmini.json")
    model_manifest.update({
        "tokenizer_included": True,
        "tokenizer_relative_path": "../tokenizer/tokenizer.json",
        "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
        "tokenizer_manifest_sha256": sha256_file(tokenizer_sidecar),
        "prepared_dataset_relative_path": "../dataset",
        "prepared_dataset_manifest_sha256": sha256_file(dataset_manifest_path),
        "train_tokens_sha256": dataset_manifest["train_tokens_sha256"],
        "valid_tokens_sha256": dataset_manifest["valid_tokens_sha256"],
        "train_mask_sha256": dataset_manifest.get("train_mask_sha256"),
        "valid_mask_sha256": dataset_manifest.get("valid_mask_sha256"),
        "corpus_sha256": dataset_manifest.get("corpus_sha256"),
        "source_provenance_sha256": dataset_manifest.get("source_provenance_sha256"),
        "release_mode": bool(dataset_manifest.get("release_mode", False)),
        "training_runtime_included": True,
        "training_runtime_enabled": False,
        "training_runtime_manifest_sha256": sha256_file(RUNTIME_DIR_LOCAL / "TRAINING_RUNTIME.json"),
    })
    write_json(manifest_path, model_manifest)

    license_note = (
        "# TinyStories English training data source and attribution\n\n"
        + "- Dataset: " + DATASET_ID + " subset " + DATASET_SUBSET + "\n"
        + "- Pinned dataset revision: " + DATASET_REVISION + "\n"
        + "- Dataset page: " + DATASET_PAGE + "\n"
        + "- Source: TinyStories dataset contributors; English synthetic stories.\n"
        + "- Declared dataset licenses on the source page: CDLA-Sharing-1.0.\n"
        + "- All four downloaded training Parquet shard hashes are recorded in BUILD_REPORT.json.\n"
        + "- Before redistributing the corpus or a model trained on it, review and comply "
          "with the applicable attribution and share-alike terms. This note is not legal advice.\n"
    )
    (LOCAL_PACKAGE / "DATA_SOURCE_AND_LICENSE.md").write_text(license_note, encoding="utf-8")

    write_json(LOCAL_PACKAGE / "BUILD_REPORT.json", {
        "format": "qmini-english-training-package-report-v1",
        "source_revision": SOURCE_REVISION,
        "initial_model_source": INITIAL_DIR_SOURCE.name,
        "initial_model_sha256": INITIAL_SHA256,
        "dataset_page": DATASET_PAGE,
        "dataset_revision": DATASET_REVISION,
        "dataset_files": data_fingerprints,
        "context_length": TRAIN_SEQUENCE_LENGTH,
        "parameters": 705742,
        "vocabulary_size": VOCAB_SIZE,
        "validation_fraction": VALIDATION_FRACTION,
        "training": {
            "target_optimizer_steps": TARGET_OPTIMIZER_STEPS,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation": GRAD_ACCUM,
            "sequence_length": TRAIN_SEQUENCE_LENGTH,
            "started": False,
        },
        "license_notice": "CDLA-Sharing-1.0; see DATA_SOURCE_AND_LICENSE.md",
    })

    verify_training_package(LOCAL_PACKAGE)
    DRIVE_STAGING = PACKAGE_ROOT / ("." + PACKAGE_NAME + ".staging-" + uuid.uuid4().hex[:8])
    try:
        shutil.rmtree(DRIVE_STAGING, ignore_errors=True)
        shutil.copytree(LOCAL_PACKAGE, DRIVE_STAGING)
        verify_training_package(DRIVE_STAGING)
        DRIVE_STAGING.rename(PACKAGE_DIR)
    except Exception:
        shutil.rmtree(DRIVE_STAGING, ignore_errors=True)
        raise
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
    shutil.rmtree(STAGE_DIR, ignore_errors=True)
    print("Training package created and Drive copy verified.")

# Re-verify persistent package every time before creating an optimizer.
PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
INITIAL_DIR = PACKAGE_DIR / "initial_brain"
INITIAL_MODEL_PATH = INITIAL_DIR / "model.qmini.npz"
CONFIG_PATH = INITIAL_DIR / "config.json"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"
DATASET_DIR = PACKAGE_DIR / "dataset"
TRAIN_BIN = DATASET_DIR / "train.bin"
VALID_BIN = DATASET_DIR / "valid.bin"
TRAIN_MASK = DATASET_DIR / "train.mask.bin"
VALID_MASK = DATASET_DIR / "valid.mask.bin"
RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
MIRROR_DIR = PACKAGE_DIR / "training_state"
PROGRESS_PATH = MIRROR_DIR / "training_progress.json"
LOCAL_RUN_DIR = Path("/content/qmini-english-512-runs") / PACKAGE_NAME

from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
    train_model,
)
from qmini.checkpointing import (
    TrainingSessionLease,
    mark_training_status,
    restore_latest_checkpoint,
)
from qmini.export import verify_packed_export

print("\n[6/8] Checking package, disk space and checkpoint safety...")
assert PACKAGE_CONFIG.max_context == 512
assert PACKAGE_CONFIG.estimated_parameter_count() == 705742
_verify_initial_brain_artifact(INITIAL_MODEL_PATH, PACKAGE_CONFIG)
_verify_prepared_dataset_manifest(
    TRAIN_BIN, VALID_BIN, TOKENIZER_PATH, TRAIN_MASK, VALID_MASK, PACKAGE_CONFIG.vocab_size
)
checkpoint_estimate_bytes = int(PACKAGE_CONFIG.estimated_parameter_count() * 12 * 1.2 + 512 * 1024**2)
drive_required_bytes = 3 * checkpoint_estimate_bytes + 1024**3
local_required_bytes = 2 * checkpoint_estimate_bytes + 1024**3
drive_free_bytes = shutil.disk_usage(MY_DRIVE).free
local_free_bytes = shutil.disk_usage("/content").free
print(f"Drive free: {drive_free_bytes / (1024**3):.2f} GiB")
print(f"Colab local free: {local_free_bytes / (1024**3):.2f} GiB")
if drive_free_bytes < drive_required_bytes:
    raise RuntimeError("Not enough Google Drive space for verified checkpoint generations and model export.")
if local_free_bytes < local_required_bytes:
    raise RuntimeError("Not enough Colab temporary disk space for safe checkpoint creation.")

progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
print("Current saved training status:", progress_now.get("status"))
print("Last verified step:", progress_now.get("current_step", 0), "/", progress_now.get("target_steps"))
print("Train tokens:", f"{len(train_tokens):,}")
print("Validation tokens:", f"{len(valid_tokens):,}")
print("Training device: CPU-only")

print("CPU threads:", CPU_THREADS)
print("Sequence length:", TRAIN_SEQUENCE_LENGTH)
print("Target optimizer steps:", TARGET_OPTIMIZER_STEPS)
print("CPU batch settings: physical batch=4 | gradient accumulation=2")

print("Effective batch=8 | tokens per optimizer step=4,096")

print("CPU mode: FP32; CUDA disabled.")


print("\n[7/8] Training/resuming the existing initial model...")
progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
if progress_now.get("status") == "completed" and progress_now.get("final_model_sha256"):
    print("This exact English dataset/model package is already fully trained. Verifying the saved final model; no duplicate training run will be started.")
    completed_step = progress_now.get("current_step")
    completed_target = progress_now.get("target_steps")
    if (
        isinstance(completed_step, bool)
        or not isinstance(completed_step, int)
        or completed_step < 1
        or completed_step != completed_target
    ):
        raise RuntimeError("Drive says training is complete, but the saved optimizer step/target is inconsistent.")
    final_drive_dir = (
        PACKAGE_DIR / "trained_models"
        / f"QMini-English-512-trained-step-{completed_step:012d}"
    )
    final_manifest_path = final_drive_dir / "model_manifest.json"
    if not final_manifest_path.is_file():
        raise FileNotFoundError("Drive progress says training is complete, but the verified final model manifest is missing.")
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    if final_manifest.get("step") != completed_step:
        raise RuntimeError("Saved final model step does not match the completed Drive progress record.")
    final_packed_path = final_drive_dir / final_manifest["packed_export"]
    final_report = verify_packed_export(final_packed_path)
    if (
        final_report.get("format") != "qmini-packed-v2"
        or final_report.get("q2_version") != 2
        or final_report.get("parameter_count") != PACKAGE_CONFIG.estimated_parameter_count()
        or sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256")
    ):
        raise RuntimeError("The previously completed final model failed Q2-v2 or SHA-256 verification.")
    _completed_digest = hashlib.sha256()
    for _item in sorted(final_drive_dir.rglob("*")):
        if _item.is_file():
            _completed_digest.update(_item.relative_to(final_drive_dir).as_posix().encode("utf-8"))
            _completed_digest.update(sha256_file(_item).encode("ascii"))
    if _completed_digest.hexdigest() != progress_now.get("final_model_sha256"):
        raise RuntimeError("The final model directory hash differs from the completed Drive progress record.")
    final_progress = progress_now
    print("Previously completed model passed final format and hash verification.")
else:
    LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
    session_started = datetime.now(timezone.utc).isoformat()
    with TrainingSessionLease(
        MIRROR_DIR,
        stale_after_seconds=LEASE_STALE_SECONDS,
        heartbeat_seconds=LEASE_HEARTBEAT_SECONDS,
    ) as lease:
        previous_progress = (
            json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
            if PROGRESS_PATH.is_file() else {}
        )
        if previous_progress.get("status") == "running":
            mark_training_status(
                MIRROR_DIR, "interrupted",
                note="Previous Colab runtime ended without releasing its training lease.",
            )

        restored = restore_latest_checkpoint(MIRROR_DIR, LOCAL_RUN_DIR)
        if restored is None:
            current_progress = (
                json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
                if PROGRESS_PATH.is_file() else {}
            )
            if current_progress.get("current_step", 0) not in (0, None):
                raise RuntimeError(
                    "Drive progress shows training steps, but no verified checkpoint can be restored. "
                    "Refusing to restart from random weights."
                )
            resume_run = False
            extend_schedule = False
            current_step = 0
        else:
            resume_run = True
            extend_schedule = TARGET_OPTIMIZER_STEPS > restored["target_steps"]
            current_step = restored["step"]
            if current_step > TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("Saved optimizer step is above this notebook's target.")
            if restored["target_steps"] > TARGET_OPTIMIZER_STEPS and current_step < TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("The new target is below the previous run's target; refusing to shorten the schedule.")

        # Keep the existing checkpoint's effective batch size unchanged.
        BATCH_SIZE = 4
        GRAD_ACCUM = 2
        GLOBAL_BATCH_SIZE = 8
        print("CPU training settings: batch=4, grad_accum=2, effective_batch=8")
        print("Tokens per optimizer step:", BATCH_SIZE * GRAD_ACCUM * TRAIN_SEQUENCE_LENGTH)

        mark_training_status(
            MIRROR_DIR, "running",
            note=f"Q-Mini English 512 training session started at {session_started}.",
        )
        print("Resume:", resume_run)
        print("Starting step:", current_step, "/", TARGET_OPTIMIZER_STEPS)

        print("Starting CPU-only training; no CUDA calls will be made.")

        FINAL_LOCAL_DIR = train_model(
            config=PACKAGE_CONFIG,
            train_bin=TRAIN_BIN,
            valid_bin=VALID_BIN,
            output_dir=LOCAL_RUN_DIR,
            steps=TARGET_OPTIMIZER_STEPS,
            batch_size=BATCH_SIZE,
            seq_len=TRAIN_SEQUENCE_LENGTH,
            grad_accum=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            eval_every=EVALUATE_EVERY,
            save_every=CHECKPOINT_EVERY,
            eval_batches=EVALUATION_BATCHES,
            resume=resume_run,
            tokenizer_path=TOKENIZER_PATH,
            extend_schedule=extend_schedule,
            train_mask_bin=TRAIN_MASK,
            valid_mask_bin=VALID_MASK,
            init_from_initial=None if resume_run else INITIAL_MODEL_PATH,
            checkpoint_mirror_dir=MIRROR_DIR,
            checkpoint_session_id=lease.session_id,
            allow_batch_size_change=False,
        )

        final_manifest_path = FINAL_LOCAL_DIR / "model_manifest.json"
        if not final_manifest_path.is_file():
            raise FileNotFoundError("Training did not produce a final_model manifest.")
        final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
        final_packed_path = FINAL_LOCAL_DIR / final_manifest["packed_export"]
        final_report = verify_packed_export(final_packed_path)
        if (
            final_report.get("format") != "qmini-packed-v2"
            or final_report.get("q2_version") != 2
            or final_report.get("parameter_count") != PACKAGE_CONFIG.estimated_parameter_count()
        ):
            raise RuntimeError("Final trained Q2-v2 export failed format/parameter verification.")
        if sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256"):
            raise RuntimeError("Final trained model SHA-256 differs from its manifest.")

        final_step = int(final_manifest["step"])
        trained_root = PACKAGE_DIR / "trained_models"
        trained_root.mkdir(parents=True, exist_ok=True)
        final_name = f"QMini-English-512-trained-step-{final_step:012d}"
        final_drive_dir = trained_root / final_name
        staging_dir = trained_root / ("." + final_name + ".staging-" + uuid.uuid4().hex[:8])

        if final_drive_dir.exists():
            existing_manifest = json.loads((final_drive_dir / "model_manifest.json").read_text(encoding="utf-8"))
            existing_packed = final_drive_dir / existing_manifest["packed_export"]
            if (
                existing_manifest.get("step") != final_step
                or existing_manifest.get("packed_export_sha256") != final_manifest.get("packed_export_sha256")
                or sha256_file(existing_packed) != existing_manifest.get("packed_export_sha256")
            ):
                raise FileExistsError("An existing trained model conflicts with this final optimizer step.")
            verify_packed_export(existing_packed)
        else:
            try:
                shutil.copytree(FINAL_LOCAL_DIR, staging_dir)
                copied_manifest = json.loads((staging_dir / "model_manifest.json").read_text(encoding="utf-8"))
                copied_packed = staging_dir / copied_manifest["packed_export"]
                if sha256_file(copied_packed) != copied_manifest.get("packed_export_sha256"):
                    raise RuntimeError("Final model Drive-copy SHA-256 verification failed.")
                verify_packed_export(copied_packed)
                staging_dir.rename(final_drive_dir)
            except Exception:
                shutil.rmtree(staging_dir, ignore_errors=True)
                raise

        directory_digest = hashlib.sha256()
        for item in sorted(final_drive_dir.rglob("*")):
            if item.is_file():
                directory_digest.update(item.relative_to(final_drive_dir).as_posix().encode("utf-8"))
                directory_digest.update(sha256_file(item).encode("ascii"))
        final_model_digest = directory_digest.hexdigest()
        final_progress = mark_training_status(
            MIRROR_DIR, "completed",
            final_model_sha256=final_model_digest,
            note=f"Verified trained Q2-v2 model stored at trained_models/{final_name}",
        )

    print("\n[8/8] Q-MINI ENGLISH TRAINING COMPLETE")
    print("Optimizer step:", final_progress["current_step"], "/", final_progress["target_steps"])
    print("Tokens seen:", f"{final_progress['tokens_seen']:,}")
    print("Trained model:", final_drive_dir)
    print("Model file:", final_drive_dir / "model.qmini.npz")
    print("Verified checkpoints:", MIRROR_DIR)
    print("Progress file:", PROGRESS_PATH)
    print("\nTraining updates were made to the saved initial model. The original initial model remains unchanged.")
